In [201]:
import psycopg2
from psycopg2 import Error 
def create_db_connection(host_name, port_name, user_name, user_password, dbname):
    connection = None
    try:

        connection = psycopg2.connect(dbname=dbname, user=user_name, password=user_password, host=host_name, port=port_name)
        print("openGauss Database connection successful")
    except Error as err:
        print(f"Error: '{err}'")

    return connection

In [202]:
def read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except Error as err:
        print(f"Error: '{err}'")

In [203]:
def execute_query(connection,query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print("Query successful")
    except Error as err:
        print(f"Error: '{err}'")

In [204]:
connection = create_db_connection("localhost", "5432", "user3", "user3_password", "db3")

openGauss Database connection successful


In [205]:
e1 = """
CREATE OR REPLACE PROCEDURE kk.proc_product1(
    IN emp_no_toBeAsked INT,
    INOUT my_cursor REFCURSOR
)
AS
BEGIN
    OPEN my_cursor FOR
    SELECT e.first_name || ' ' || e.last_name AS full_name, e.birth_date, e.gender, s.salary, s.from_date, s.to_date
    FROM kk.employees e
    JOIN kk.salaries s ON e.emp_no = s.emp_no
    WHERE e.emp_no = emp_no_toBeAsked;
END;
"""
execute_query(connection,e1.encode(encoding='utf-8'))


execute_query(connection, "BEGIN;".encode(encoding='utf-8'))

r2 = """
CALL kk.proc_product1(10001, 'mycursor');
FETCH ALL FROM mycursor;
"""
results2 = read_query(connection, r2)

for result2 in results2:
    print(result2)
execute_query(connection, "COMMIT;".encode(encoding='utf-8'))

Query successful
Query successful
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', 60117, datetime.datetime(1986, 6, 26, 0, 0), datetime.datetime(1987, 6, 26, 0, 0))
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', 62102, datetime.datetime(1987, 6, 26, 0, 0), datetime.datetime(1988, 6, 25, 0, 0))
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', 66074, datetime.datetime(1988, 6, 25, 0, 0), datetime.datetime(1989, 6, 25, 0, 0))
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', 66596, datetime.datetime(1989, 6, 25, 0, 0), datetime.datetime(1990, 6, 25, 0, 0))
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', 66961, datetime.datetime(1990, 6, 25, 0, 0), datetime.datetime(1991, 6, 25, 0, 0))
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', 71046, datetime.datetime(1991, 6, 25, 0, 0), datetime.datetime(1992, 6, 24, 0, 0))
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', 74333, datetime.datetime(1992, 6, 24, 0, 

In [206]:
e2 = """
CREATE OR REPLACE PROCEDURE kk.proc_product2(
    p_emp_no IN INT,                  -- 输入参数：员工编号
    o_name OUT VARCHAR(31),     -- 输出参数1：名字
    o_birth_date OUT DATE,            -- 输出参数2：生日
    o_gender OUT VARCHAR(6),          -- 输出参数3：性别
    o_salaries OUT TEXT[]             -- 输出参数4：工资数组（使用内置数组类型）
) 
AS
BEGIN
    -- 查询基础信息
    SELECT first_name || ' ' || last_name, birth_date, gender 
    INTO o_name, o_birth_date, o_gender
    FROM kk.employees 
    WHERE emp_no = p_emp_no;

    -- 聚合工资记录为数组
    SELECT ARRAY_AGG(salary || ',' || from_date || ',' || to_date)
    INTO o_salaries
    FROM kk.salaries
    WHERE emp_no = p_emp_no;
END;
"""
execute_query(connection,e2.encode(encoding='utf-8'))


r3 = """
CALL kk.proc_product2(10001,o_name,o_birth_date,o_gender,o_salaries);
"""
results3 = read_query(connection, r3)

for result3 in results3:
    print(result3)

Query successful
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', ['60117,1986-06-26 00:00:00,1987-06-26 00:00:00', '62102,1987-06-26 00:00:00,1988-06-25 00:00:00', '66074,1988-06-25 00:00:00,1989-06-25 00:00:00', '66596,1989-06-25 00:00:00,1990-06-25 00:00:00', '66961,1990-06-25 00:00:00,1991-06-25 00:00:00', '71046,1991-06-25 00:00:00,1992-06-24 00:00:00', '74333,1992-06-24 00:00:00,1993-06-24 00:00:00', '75286,1993-06-24 00:00:00,1994-06-24 00:00:00', '75994,1994-06-24 00:00:00,1995-06-24 00:00:00', '76884,1995-06-24 00:00:00,1996-06-23 00:00:00', '80013,1996-06-23 00:00:00,1997-06-23 00:00:00', '81025,1997-06-23 00:00:00,1998-06-23 00:00:00', '81097,1998-06-23 00:00:00,1999-06-23 00:00:00', '84917,1999-06-23 00:00:00,2000-06-22 00:00:00', '85112,2000-06-22 00:00:00,2001-06-22 00:00:00', '85097,2001-06-22 00:00:00,2002-06-22 00:00:00', '88958,2002-06-22 00:00:00,9999-01-01 00:00:00'])


In [207]:
execute_query(connection, "DROP PROCEDURE kk.proc_product2;".encode(encoding='utf-8'))

Query successful


In [208]:
e3 = """
CREATE OR REPLACE FUNCTION kk.get_employee_detail(p_emp_no INT)
RETURNS TABLE (
    o_dept_name VARCHAR(40),
    o_title VARCHAR(50),
    o_salary INT
)
AS $$
BEGIN
    RETURN QUERY
    SELECT d.dept_name,t.title,s.salary
    FROM kk.titles t
    JOIN kk.dept_emp de ON de.emp_no = t.emp_no
    JOIN kk.departments d ON d.dept_no = de.dept_no
    JOIN kk.salaries s ON s.emp_no = de.emp_no
    WHERE de.emp_no = p_emp_no;
END;
$$ LANGUAGE PLPGSQL;

"""
execute_query(connection,e3.encode(encoding='utf-8'))

r4 = """
SELECT * FROM kk.get_employee_detail(10001);
"""
results4 = read_query(connection, r4)

for result4 in results4:
    print(result4)

Query successful
('Development', 'Senior Engineer', 60117)
('Development', 'Senior Engineer', 62102)
('Development', 'Senior Engineer', 66074)
('Development', 'Senior Engineer', 66596)
('Development', 'Senior Engineer', 66961)
('Development', 'Senior Engineer', 71046)
('Development', 'Senior Engineer', 74333)
('Development', 'Senior Engineer', 75286)
('Development', 'Senior Engineer', 75994)
('Development', 'Senior Engineer', 76884)
('Development', 'Senior Engineer', 80013)
('Development', 'Senior Engineer', 81025)
('Development', 'Senior Engineer', 81097)
('Development', 'Senior Engineer', 84917)
('Development', 'Senior Engineer', 85112)
('Development', 'Senior Engineer', 85097)
('Development', 'Senior Engineer', 88958)


In [209]:
e4 = """
CREATE OR REPLACE FUNCTION kk.INSERT_Func(
    p_dept_no CHAR(4),
    p_dept_name VARCHAR(40))
RETURNS INT
AS $$
DECLARE
    dept_count INT;
BEGIN
    -- 插入新部门
    INSERT INTO kk.departments(dept_no, dept_name)
    VALUES (p_dept_no, p_dept_name);

    -- 获取最新部门数
    SELECT COUNT(*) INTO dept_count FROM kk.departments;
    
    RETURN dept_count;
EXCEPTION
    WHEN unique_violation THEN
        RAISE NOTICE '部门已存在';
        RETURN -1;
END;
$$ LANGUAGE PLPGSQL;
"""
execute_query(connection,e4.encode(encoding='utf-8'))

r5 = """
SELECT kk.INSERT_Func('d010', 'New Department');
"""
results5 = read_query(connection, r5)

for result5 in results5:
    print(result5)

Query successful
(10,)


In [210]:
e5 = """
CREATE OR REPLACE FUNCTION kk.departments_trigger_func()
RETURNS TRIGGER
AS $$
BEGIN
    NEW.dept_name := 'ceshi';
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;
"""
execute_query(connection,e5.encode(encoding='utf-8'))

e6 = """
CREATE TRIGGER before_departments_insert
BEFORE INSERT ON kk.departments
FOR EACH ROW
EXECUTE PROCEDURE kk.departments_trigger_func();
"""
execute_query(connection,e6.encode(encoding='utf-8'))



Query successful
Query successful


In [211]:
e7 = """
CREATE OR REPLACE FUNCTION kk.FUNC1(p_emp_no INT)
RETURNS TABLE (
    full_name VARCHAR(31),
    birth_date DATE,
    gender VARCHAR(6),
    salaries TEXT[]
)
AS $$
BEGIN
    -- 获取基础信息
    SELECT e.first_name || ' ' || e.last_name, 
           e.birth_date, 
           e.gender
    INTO full_name, birth_date, gender
    FROM kk.employees e
    WHERE e.emp_no = p_emp_no;

    -- 聚合工资记录为文本数组
    SELECT ARRAY_AGG(
        salary::TEXT || ',' || from_date::TEXT || ',' || to_date::TEXT
    ) INTO salaries
    FROM kk.salaries
    WHERE emp_no = p_emp_no;

    RETURN NEXT;
END;
$$ LANGUAGE PLPGSQL;
"""
execute_query(connection,e7.encode(encoding='utf-8'))


r6 = """
SELECT * FROM kk.FUNC1(10001);
"""
results6 = read_query(connection, r6)

for result6 in results6:
    print(result6)

Query successful
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', ['60117,1986-06-26 00:00:00,1987-06-26 00:00:00', '62102,1987-06-26 00:00:00,1988-06-25 00:00:00', '66074,1988-06-25 00:00:00,1989-06-25 00:00:00', '66596,1989-06-25 00:00:00,1990-06-25 00:00:00', '66961,1990-06-25 00:00:00,1991-06-25 00:00:00', '71046,1991-06-25 00:00:00,1992-06-24 00:00:00', '74333,1992-06-24 00:00:00,1993-06-24 00:00:00', '75286,1993-06-24 00:00:00,1994-06-24 00:00:00', '75994,1994-06-24 00:00:00,1995-06-24 00:00:00', '76884,1995-06-24 00:00:00,1996-06-23 00:00:00', '80013,1996-06-23 00:00:00,1997-06-23 00:00:00', '81025,1997-06-23 00:00:00,1998-06-23 00:00:00', '81097,1998-06-23 00:00:00,1999-06-23 00:00:00', '84917,1999-06-23 00:00:00,2000-06-22 00:00:00', '85112,2000-06-22 00:00:00,2001-06-22 00:00:00', '85097,2001-06-22 00:00:00,2002-06-22 00:00:00', '88958,2002-06-22 00:00:00,9999-01-01 00:00:00'])


In [212]:
e8 = """
CREATE OR REPLACE FUNCTION kk.FUNC2(p_emp_no INT)
RETURNS TABLE (
    full_name VARCHAR(31),
    birth_date DATE,
    gender VARCHAR(6),
    avg_salary NUMERIC
)
AS $$
DECLARE
    salary_records TEXT[];
BEGIN
    -- 获取FUNC1的结果
    SELECT f.salaries INTO salary_records
    FROM kk.FUNC1(p_emp_no) f;

    -- 计算平均工资
    SELECT AVG(
        (string_to_array(unnest, ','))[1]::NUMERIC
    ) INTO avg_salary
    FROM unnest(salary_records);

    -- 获取基础信息
    SELECT f.full_name, f.birth_date, f.gender
    INTO full_name, birth_date, gender
    FROM kk.FUNC1(p_emp_no) f;

    RETURN NEXT;
END;
$$ LANGUAGE PLPGSQL;

"""
execute_query(connection,e8.encode(encoding='utf-8'))

r7 = """
SELECT * FROM kk.FUNC2(10001);
"""
results7 = read_query(connection, r7)

for result7 in results7:
    print(result7)


Query successful
('Georgi Facello', datetime.datetime(1953, 9, 2, 0, 0), 'M', Decimal('75388.941176470588'))


In [213]:
e9 = """
CREATE OR REPLACE FUNCTION kk.employees_delete_trigger_func()
RETURNS TRIGGER
AS $$
BEGIN
    -- 级联删除相关表记录
    DELETE FROM kk.dept_manager WHERE emp_no = OLD.emp_no;
    DELETE FROM kk.dept_emp WHERE emp_no = OLD.emp_no;
    DELETE FROM kk.salaries WHERE emp_no = OLD.emp_no;
    DELETE FROM kk.titles WHERE emp_no = OLD.emp_no;
    RETURN OLD;
END;
$$ LANGUAGE PLPGSQL;

"""
execute_query(connection,e9.encode(encoding='utf-8'))

e10 = """
CREATE TRIGGER employees_delete_trigger
BEFORE DELETE ON kk.employees
FOR EACH ROW
EXECUTE PROCEDURE kk.employees_delete_trigger_func();

"""
execute_query(connection,e10.encode(encoding='utf-8'))

Query successful
Query successful


In [214]:
e11 = """
CREATE OR REPLACE FUNCTION kk.insert_or_update_sal_trigger_func()
RETURNS TRIGGER
AS $$
BEGIN
    -- 工资校验逻辑
    IF NEW.salary < 80000 THEN
        NEW.salary := 80000;
        RAISE NOTICE '员工 % 的工资已自动调整为 80000', NEW.emp_no;
    END IF;
    RETURN NEW;
END;
$$ LANGUAGE PLPGSQL;

"""
execute_query(connection,e11.encode(encoding='utf-8'))

e12 = """
CREATE TRIGGER insert_or_update_sal
BEFORE INSERT OR UPDATE ON kk.salaries
FOR EACH ROW
EXECUTE PROCEDURE kk.insert_or_update_sal_trigger_func();

"""
execute_query(connection,e12.encode(encoding='utf-8'))

Query successful
Query successful


In [215]:
e13 = """
INSERT INTO kk.salaries VALUES 
(10005, 80001, '1995-06-24', '1996-06-24'),
(10006, 78888, '2000-06-22', '2002-09-22');

"""
execute_query(connection,e13.encode(encoding='utf-8'))

r1 = """
SELECT * FROM kk.salaries WHERE emp_no IN (10005, 10006);
"""

results = read_query(connection, r1)

for result in results:
    print(result)

Query successful
(10005, 78228, datetime.datetime(1989, 9, 12, 0, 0), datetime.datetime(1990, 9, 12, 0, 0))
(10005, 82621, datetime.datetime(1990, 9, 12, 0, 0), datetime.datetime(1991, 9, 12, 0, 0))
(10005, 83735, datetime.datetime(1991, 9, 12, 0, 0), datetime.datetime(1992, 9, 11, 0, 0))
(10005, 85572, datetime.datetime(1992, 9, 11, 0, 0), datetime.datetime(1993, 9, 11, 0, 0))
(10005, 85076, datetime.datetime(1993, 9, 11, 0, 0), datetime.datetime(1994, 9, 11, 0, 0))
(10005, 86050, datetime.datetime(1994, 9, 11, 0, 0), datetime.datetime(1995, 9, 11, 0, 0))
(10005, 88448, datetime.datetime(1995, 9, 11, 0, 0), datetime.datetime(1996, 9, 10, 0, 0))
(10005, 88063, datetime.datetime(1996, 9, 10, 0, 0), datetime.datetime(1997, 9, 10, 0, 0))
(10005, 89724, datetime.datetime(1997, 9, 10, 0, 0), datetime.datetime(1998, 9, 10, 0, 0))
(10005, 90392, datetime.datetime(1998, 9, 10, 0, 0), datetime.datetime(1999, 9, 10, 0, 0))
(10005, 90531, datetime.datetime(1999, 9, 10, 0, 0), datetime.datetime(20

In [216]:
e14 = """
DROP TRIGGER IF EXISTS insert_or_update_sal ON kk.salaries;
"""
execute_query(connection,e14.encode(encoding='utf-8'))

Query successful
